# KoHRM-Text-1.4B Colab T4 Generation Test

This notebook is designed for a Google Colab T4 runtime. It downloads the latest public KoHRM-Text export, loads the tokenizer and `model.safetensors`, and runs a short generation test.

Important notes:

- This notebook does **not** import `transformers`, so it avoids the common Colab `torchvision::nms` / custom architecture import failure.
- It uses `tokenizers.Tokenizer.from_file` for tokenization.
- It uses `kohrm_colab_generate.py`, a small PyTorch SDPA runtime for the HRM-Text architecture.
- First run downloads the model weights, roughly 2.8 GiB, and loads them in fp16 on GPU.

한국어 요약: 이 노트북은 Colab T4에서 바로 최신 공개 KoHRM-Text 모델을 받아 짧은 생성을 실행합니다. `transformers`를 쓰지 않고, 공개 `model.safetensors`를 직접 읽는 전용 PyTorch 추론 경로를 사용합니다.


## 1. Install dependencies

Colab already includes PyTorch. This installs only the Hub/tokenizer/safetensors packages needed by the notebook.


In [ ]:
!pip -q install -U huggingface_hub hf_transfer tokenizers safetensors

## 2. Runtime and download settings

Keep `RUN_GENERATION=True` for the actual generation test. Set it to `False` only when you want a fast tokenizer/config check.


In [ ]:
import os
import json
import math
import subprocess
import sys
import importlib.util
from pathlib import Path

import torch
from huggingface_hub import HfApi, snapshot_download
from tokenizers import Tokenizer
from safetensors import safe_open

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B"
REVISION = "main"
RUN_GENERATION = True
MAX_SEQ_LEN = 512
MAX_NEW_TOKENS = 64 if torch.cuda.is_available() else 8
TEMPERATURE = 0.0
TOP_P = 0.9

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print("gpu memory GiB:", round(free / 2**30, 2), "/", round(total / 2**30, 2))

info = HfApi().model_info(REPO_ID, revision=REVISION)
print("latest hub sha:", info.sha)

## 3. Download the latest public files

The generation helper is downloaded from the model repo when available. If it is missing, the notebook falls back to cloning the project repo inside Colab.


In [ ]:
patterns = [
    "README.md",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model.safetensors",
    "kohrm_colab_generate.py",
]

repo_dir = Path(snapshot_download(
    repo_id=REPO_ID,
    revision=REVISION,
    allow_patterns=patterns,
    max_workers=8,
))

print("downloaded to:", repo_dir)
print("files:")
for path in sorted(repo_dir.iterdir()):
    if path.is_file():
        print(" -", path.name, round(path.stat().st_size / 2**20, 2), "MiB")

helper_path = repo_dir / "kohrm_colab_generate.py"
if not helper_path.exists():
    project_dir = Path("/content/KoHRM-text")
    if not project_dir.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/LLM-OS-Models/KoHRM-text",
            str(project_dir),
        ], check=True)
    helper_path = project_dir / "notebooks" / "kohrm_colab_generate.py"

print("generation helper:", helper_path)
assert helper_path.exists(), helper_path

## 4. Load config and tokenizer

This intentionally uses the low-level tokenizer package, not `AutoTokenizer`.


In [ ]:
config = json.loads((repo_dir / "config.json").read_text())
print(json.dumps({
    "model_type": config.get("model_type"),
    "architectures": config.get("architectures"),
    "vocab_size": config.get("vocab_size"),
    "hidden_size": config.get("hidden_size"),
    "num_hidden_layers": config.get("num_hidden_layers"),
    "num_attention_heads": config.get("num_attention_heads"),
    "max_position_embeddings": config.get("max_position_embeddings"),
    "prefix_lm": config.get("prefix_lm"),
}, indent=2, ensure_ascii=False))

tokenizer = Tokenizer.from_file(str(repo_dir / "tokenizer.json"))
special_tokens = [
    "<|im_start|>",
    "<|im_end|>",
    "<|box_end|>",
    "<|object_ref_start|>",
    "<|object_ref_end|>",
    "<|quad_start|>",
    "<|quad_end|>",
]
special_token_ids = {tok: tokenizer.token_to_id(tok) for tok in special_tokens}

print("tokenizer vocab size:", tokenizer.get_vocab_size())
print("special token ids:")
print(json.dumps(special_token_ids, indent=2, ensure_ascii=False))

## 5. Tokenizer experiments

This checks Korean, terminal, JSON/tool-call, and code prompts with the project prompt wrapper.


In [ ]:
def format_prompt(prompt: str, condition_token: str = "<|object_ref_start|>") -> str:
    return f"<|im_start|>{condition_token}{prompt}<|im_end|>"

samples = {
    "korean_terminal": "한국어로 현재 디렉터리에서 용량이 큰 파일 20개를 찾고, .data 폴더는 별도로 합산하는 bash 명령을 작성하세요.",
    "tool_call_json": '{"tool":"exec_command","arguments":{"cmd":"du -h --max-depth=1 .data | sort -h","workdir":"/home/work"}}',
    "python_code": "def top_k_files(root, k=20):
    return sorted(root.rglob('*'), key=lambda p: p.stat().st_size, reverse=True)[:k]",
    "english": "Explain why PrefixLM can use bidirectional attention over the input prefix while preserving causal generation over the response.",
}

rows = []
for name, text in samples.items():
    wrapped = format_prompt(text)
    ids = tokenizer.encode(wrapped).ids
    rows.append((name, len(text), len(ids), round(len(text) / max(1, len(ids)), 2), ids[:16]))

print(f"{'name':<18} {'chars':>8} {'tokens':>8} {'chars/token':>12} first_ids")
for row in rows:
    print(f"{row[0]:<18} {row[1]:>8} {row[2]:>8} {row[3]:>12} {row[4]}")

## 6. Inspect `model.safetensors` before generation

This reads tensor metadata first. It confirms that the weight file is complete before the full fp16 GPU load.


In [ ]:
weights = repo_dir / "model.safetensors"
assert weights.exists(), "model.safetensors was not downloaded"

with safe_open(weights, framework="pt", device="cpu") as f:
    keys = list(f.keys())
    total_params = 0
    preview = []
    for key in keys:
        shape = tuple(f.get_slice(key).get_shape())
        n = math.prod(shape)
        total_params += n
        if len(preview) < 12:
            preview.append((key, shape, n))

print("num tensors:", len(keys))
print("num params:", f"{total_params:,}")
print("fp16/bf16 weight size estimate GiB:", round(total_params * 2 / 2**30, 2))
print("first tensors:")
for key, shape, n in preview:
    print(" -", key, shape, f"{n:,}")

## 7. Run text generation

This imports the project-side lightweight runtime and runs greedy generation. On a free Colab T4, the first call can take a few minutes because it loads all weights and initializes the KV cache.


In [ ]:
if RUN_GENERATION:
    spec = importlib.util.spec_from_file_location("kohrm_colab_generate", helper_path)
    kohrm = importlib.util.module_from_spec(spec)
    sys.modules["kohrm_colab_generate"] = kohrm
    spec.loader.exec_module(kohrm)

    prompt = "한국어로 현재 디렉터리에서 가장 큰 파일 10개를 찾는 bash 명령을 알려주세요. 명령만 간단히 답하세요."
    print("prompt:", prompt)
    print("max_new_tokens:", MAX_NEW_TOKENS, "max_seq_len:", MAX_SEQ_LEN)

    output = kohrm.generate_text(
        repo_dir,
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        max_seq_len=MAX_SEQ_LEN,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    print("
=== KoHRM output ===")
    print(output)
else:
    print("RUN_GENERATION=False, generation skipped.")

## 8. What this notebook proves

- The latest Hub revision is visible from Colab.
- Tokenizer/config files are loadable without `transformers`.
- `model.safetensors` is structurally readable.
- A short generation call runs through the KoHRM-specific PyTorch SDPA runtime.

Limitations:

- This is a smoke generation path, not a throughput benchmark.
- Plain `AutoModelForCausalLM.generate()` is still not the supported path until a full Hugging Face remote-code wrapper is added.
- CPU generation is technically possible but very slow; use a T4 or better GPU.
